# HybridKla-derived human-to-mouse benchmark

Table S1 human sites form development data; Table S7 mouse sites are a paper-designated external source. Reference sequences, labels, exact windows, and full-protein similarity are checked before locking partitions. Unreported lysines are **putative** negatives. Neither supplementary table gives per-site PMID, so independent study provenance cannot be verified at site level. This benchmark is distinct from the authors' original workbook and its reported scores.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, urllib.request, zipfile

REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
REPO_BRANCH = 'hybridkla-benchmark'
REPO_DIR = Path('/kaggle/working/Lysine-Lactylation-Sites-Prediction')
PROJECT = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT = Path('/kaggle/working/hybridkla_study_results')
SOURCE = OUTPUT / 'source'
SOURCE.mkdir(parents=True, exist_ok=True)
RUN_PAIRED_MODELS = False
RUN_HYBRIDKLA_FULL = False
SEED = 42
EPOCHS = 30
PATIENCE = 6
# Set these to attached Kaggle dataset files when the pinned online downloads are unavailable.
LOCAL_SUPPLEMENT_ZIP = None
LOCAL_HUMAN_FASTA = None
LOCAL_MOUSE_FASTA = None
LOCAL_HYBRIDKLA_ARCHIVE = None


## Repository and environment

A private GitHub token is read from the Kaggle secret `GITHUB_TOKEN`; it is not placed in the clone URL or printed. The checked-out commit and GPU are recorded.


In [ ]:
if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        clone = subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], env=env, capture_output=True, text=True)
        del token, env
    if clone.returncode:
        raise RuntimeError('Private repository clone failed; check token permissions and branch')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO_DIR, text=True).strip()
if branch != REPO_BRANCH:
    raise RuntimeError(f'Expected {REPO_BRANCH}, found {branch}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU runtime before training')
runtime = {'commit': commit, 'branch': branch, 'gpu': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'torch': torch.__version__}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')
print(json.dumps(runtime, indent=2))


## Pinned sources and candidate inventory

The checksums are verified by `prepare_hybridkla_study.py`. A changed upstream reference release requires a new audit, not silent substitution.


In [ ]:
def obtain(local, filename, url):
    destination = SOURCE / filename
    if local is not None:
        path = Path(local)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    if not destination.is_file():
        urllib.request.urlretrieve(url, destination)
    return destination

supplement = obtain(LOCAL_SUPPLEMENT_ZIP, 'PMC12309240_SupplementaryFiles.zip', 'https://www.ebi.ac.uk/europepmc/webservices/rest/PMC12309240/supplementaryFiles?inlineImages=false')
with zipfile.ZipFile(supplement) as archive:
    for filename in ('table_s1_bbaf375.xlsx', 'table_s7_bbaf375.xlsx'):
        destination = SOURCE / filename
        destination.write_bytes(archive.read(filename))
human = obtain(LOCAL_HUMAN_FASTA, 'UP000005640_9606.fasta.gz', 'https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/reference_proteomes/Eukaryota/UP000005640/UP000005640_9606.fasta.gz')
mouse = obtain(LOCAL_MOUSE_FASTA, 'UP000000589_10090.fasta.gz', 'https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/reference_proteomes/Eukaryota/UP000000589/UP000000589_10090.fasta.gz')
def command(args, log=None):
    result = subprocess.run([sys.executable, *args], cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if log:
        Path(log).write_text(result.stdout)
    if result.returncode:
        raise RuntimeError('Command failed: ' + ' '.join(args) + '\n' + result.stdout[-4000:])
    return result.stdout

prepared = OUTPUT / 'preparation'
prepared.mkdir(exist_ok=True)
print(command(['scripts/prepare_hybridkla_study.py', '--s1', str(SOURCE / 'table_s1_bbaf375.xlsx'), '--s7', str(SOURCE / 'table_s7_bbaf375.xlsx'), '--human-fasta', str(human), '--mouse-fasta', str(mouse), '--output-dir', str(prepared)]))


## Full-protein homology lock

MMseqs2 screens the eligible proteins at 30% identity and at least 80% coverage in both directions. Groups containing both human and mouse proteins are excluded. Exact windows are deduplicated. Training negatives are sampled only after splitting; validation and tests keep their full putative-negative pools.


In [ ]:
mmseqs = shutil.which('mmseqs')
if mmseqs is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'mmseqs2'], check=True)
    mmseqs = shutil.which('mmseqs')
if mmseqs is None:
    raise RuntimeError('MMseqs2 is required to establish the protein-level lock')
version = subprocess.check_output([mmseqs, 'version'], text=True).strip()
fasta = prepared / 'candidate_proteins.fasta'
hits = prepared / 'all_vs_all.tsv'
search = [mmseqs, 'easy-search', str(fasta), str(fasta), str(hits), str(prepared / 'mmseqs_tmp'), '--min-seq-id', '0.3', '-c', '0.8', '--cov-mode', '0', '-s', '7', '--max-seqs', '5000', '--threads', '4', '--format-output', 'query,target,pident,qcov,tcov', '-v', '0']
with (prepared / 'mmseqs.log').open('w') as handle:
    subprocess.run(search, check=True, stdout=handle, stderr=subprocess.STDOUT)
locked = OUTPUT / 'locked'
print(command(['scripts/lock_hybridkla_study.py', '--candidate-csv', str(prepared / 'candidate_sites.csv'), '--candidate-fasta', str(fasta), '--preparation-audit', str(prepared / 'preparation_audit.json'), '--homology-hits-tsv', str(hits), '--output-dir', str(locked), '--seed', str(SEED)]))
(OUTPUT / 'search_provenance.json').write_text(json.dumps({'version': version, 'command': search, 'commit': commit}, indent=2) + '\n')
lock = locked / 'split_audit.json'
audit = json.loads(lock.read_text())
print('External positive sites:', audit['partitions_before_training_resampling']['external_test']['positives'])
print(command(['scripts/run_replication.py', '--run', 'hybridkla-study-check', '--hybridkla-study-lock', str(lock)]))


## Matched PCBert-Kla and MM-Kla runs

Both models use the identical locked manifest, ProtBert checkpoint, SGD, train/validation partitions, selected seed and 0.5 decision threshold. Validation AUPRC selects the checkpoint. Tests are evaluated only after selection. Training is disabled by default.


In [ ]:
def paired_job(architecture, device):
    destination = OUTPUT / f'{architecture}_seed_{SEED}'
    destination.mkdir(exist_ok=True)
    args = ['scripts/run_replication.py', '--run', 'hybridkla-study', '--hybridkla-study-lock', str(lock), '--architecture', architecture, '--model-name', 'Rostlab/prot_bert', '--optimizer', 'sgd', '--learning-rate', '0.003', '--weight-decay', '0', '--scheduler', 'none', '--epochs', str(EPOCHS), '--patience', str(PATIENCE), '--batch-size', '4', '--device', device, '--seed', str(SEED), '--output-dir', str(destination)]
    logfile = (destination / 'train.log').open('w')
    process = subprocess.Popen([sys.executable, *args], cwd=PROJECT, stdout=logfile, stderr=subprocess.STDOUT)
    return process, logfile, destination

if RUN_PAIRED_MODELS:
    jobs = {}
    try:
        for architecture in ('baseline', 'token_gated'):
            device = 'cuda:0' if architecture == 'baseline' or torch.cuda.device_count() == 1 else 'cuda:1'
            jobs[architecture] = paired_job(architecture, device)
            if torch.cuda.device_count() == 1:
                process, logfile, destination = jobs.pop(architecture)
                code = process.wait()
                logfile.close()
                if code:
                    raise RuntimeError(architecture + ' failed; see ' + str(destination / 'train.log'))
        for architecture, (process, logfile, destination) in list(jobs.items()):
            code = process.wait()
            logfile.close()
            del jobs[architecture]
            if code:
                raise RuntimeError(architecture + ' failed; see ' + str(destination / 'train.log'))
    finally:
        for process, logfile, destination in jobs.values():
            if process.poll() is None:
                process.terminate()
                process.wait()
            logfile.close()
    print('Matched models completed')
else:
    print('Matched training disabled')


## HybridKla architecture-aligned retraining

This runner uses the authors' eight-branch architecture but changes the evaluation protocol to grouped out-of-fold training and the locked split. It is not an exact reproduction of their published CV. Its ESM2 branch is fine-tuned, and the full run can be expensive.


In [ ]:
if RUN_HYBRIDKLA_FULL:
    archive_path = obtain(LOCAL_HYBRIDKLA_ARCHIVE, 'HybridKla.zip', 'https://transkla.zzu.edu.cn/api/download/HybridKla.zip')
    archive_hash = hashlib.sha256(Path(archive_path).read_bytes()).hexdigest()
    if archive_hash != '73ddb128c441d7b213c4636e6bb6ce34501e82143c2b9bc06ec329d68a422993':
        raise ValueError('HybridKla resource archive checksum mismatch')
    resources = SOURCE / 'HybridKla' / 'train'
    resources.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for filename in ('top10.txt', 'BLOSUM62R.txt'):
            (resources / filename).write_bytes(archive.read('HybridKla/train/' + filename))
    destination = OUTPUT / f'hybridkla_architecture_seed_{SEED}'
    destination.mkdir(exist_ok=True)
    args = ['scripts/run_hybridkla_architecture.py', '--benchmark-lock', str(lock), '--resource-dir', str(resources), '--output-dir', str(destination), '--device', 'cuda:0', '--seed', str(SEED)]
    with (destination / 'train.log').open('w') as handle:
        result = subprocess.run([sys.executable, *args], cwd=PROJECT, stdout=handle, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError('HybridKla architecture run failed; see ' + str(destination / 'train.log'))
    print(json.dumps(json.loads((destination / 'run_report.json').read_text())['test_metrics'], indent=2))
else:
    print('HybridKla retraining disabled')


## Generated comparison

Scores below are read from completed run artifacts. Missing runs are not filled with historical or published values.


In [ ]:
import pandas as pd
records = []
for architecture in ('baseline', 'token_gated'):
    runs = OUTPUT / f'{architecture}_seed_{SEED}' / 'hybridkla_study' / 'runs'
    if not runs.exists():
        continue
    candidates = sorted(runs.glob('*/run_config.json'))
    matches = [path.parent for path in candidates if json.loads(path.read_text()).get('architecture') == architecture]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one completed {architecture} run')
    for split, filename in (('internal_test', 'test_metrics.json'), ('external_test', 'external_mouse_metrics.json')):
        records.append({'architecture': architecture, 'split': split, 'seed': SEED, **json.loads((matches[0] / filename).read_text())})
hybrid_report = OUTPUT / f'hybridkla_architecture_seed_{SEED}' / 'run_report.json'
if hybrid_report.is_file():
    for split, metrics in json.loads(hybrid_report.read_text())['test_metrics'].items():
        records.append({'architecture': 'HybridKla architecture-aligned', 'split': split, 'seed': SEED, **metrics})
summary = pd.DataFrame.from_records(records)
if summary.empty:
    print('No completed model evaluations')
else:
    display(summary)
    summary.to_csv(OUTPUT / 'matched_comparison.csv', index=False)


## Mouse external-test uncertainty

The interval below resamples whole proteins, not individual sites. It is descriptive uncertainty for the retained mouse cohort; it is not a basis for selecting a model or threshold.


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

prediction_paths = {}
for architecture in ('baseline', 'token_gated'):
    directory = OUTPUT / f'{architecture}_seed_{SEED}' / 'hybridkla_study' / 'runs'
    if directory.exists():
        files = sorted(directory.glob('*/external_mouse_predictions.csv'))
        if len(files) != 1:
            raise RuntimeError('Expected one external prediction file for ' + architecture)
        prediction_paths[architecture] = files[0]
hybrid_predictions = OUTPUT / f'hybridkla_architecture_seed_{SEED}' / 'external_test_predictions.csv'
if hybrid_predictions.is_file():
    prediction_paths['HybridKla architecture-aligned'] = hybrid_predictions
intervals = []
for architecture, path in prediction_paths.items():
    frame = pd.read_csv(path)
    if 'true_label' in frame:
        proteins = frame['name'].astype(str).to_numpy()
        labels = frame['true_label'].to_numpy(dtype=int)
        scores = frame['score'].to_numpy(dtype=float)
    else:
        proteins = frame['Site_ID'].astype(str).str.rsplit(':', n=1).str[0].to_numpy()
        labels = frame['Label'].to_numpy(dtype=int)
        scores = frame['Score'].to_numpy(dtype=float)
    unique = np.unique(proteins)
    index = {protein: np.flatnonzero(proteins == protein) for protein in unique}
    rng = np.random.default_rng(2026)
    bootstrap = []
    for _ in range(2000):
        chosen = rng.choice(unique, size=len(unique), replace=True)
        selected = np.concatenate([index[protein] for protein in chosen])
        if len(np.unique(labels[selected])) < 2:
            continue
        bootstrap.append((average_precision_score(labels[selected], scores[selected]), roc_auc_score(labels[selected], scores[selected])))
    if len(bootstrap) < 1900:
        raise RuntimeError('Too many degenerate protein-bootstrap samples')
    bounds = np.percentile(np.asarray(bootstrap), [2.5, 97.5], axis=0)
    intervals.append({'architecture': architecture, 'proteins': len(unique), 'positive_sites': int(labels.sum()), 'AUPRC_low': bounds[0, 0], 'AUPRC_high': bounds[1, 0], 'AUC_low': bounds[0, 1], 'AUC_high': bounds[1, 1]})
if intervals:
    display(pd.DataFrame(intervals))
    pd.DataFrame(intervals).to_csv(OUTPUT / 'mouse_protein_bootstrap_intervals.csv', index=False)
else:
    print('No external prediction files')


## Interpretation boundary

This split answers exact-duplicate and qualifying full-protein homology concerns for the retained records. It does not verify S1/S7 per-site study provenance or turn unreported lysines into confirmed negatives. The mouse external set is small after screening; report counts and confidence intervals, and do not use it for tuning.
